# Neural Networks II: Deeper into model building, training and optimization.

So far, we have built neural networks **completely from scratch**:

- We defined weights and biases as tensors.
- We computed predictions manually.
- We calculated the cost function by hand.
- We updated the parameters with gradient descent.


This is a great way to **learn how things work under the hood**. But in the real world, writing everything manually is slow, error-prone, and hard to scale. This is how defining a model through `nn.Module` in `PyTorch` will help us a lot!

By the end of this class, you will be able to:

- Understand PyTorch's `nn.Module` abstraction.
- Build a multi-layer perceptron (MLP) using `nn.Module`.
- Combine `nn.Module` with the built-in layers, cost functions and optimizers in `PyTorch`. 
- Train and evaluate a model.

In [ ]:
# Imports for the class
import numpy as np
import torch
import torch.nn as nn

## 1. Defining a model through `nn.Module`

`nn.Module` is a class that provides flexibility to build more complex models. `nn.Module` is the **base class for all neural network components** in PyTorch. 

Importantly, when a model is defined through the `nn.Module`, it automatically **registers its parameters (i.e., the weights and intercepts)**, tracks them for gradient updates, and provides useful methods such as `.to(device)`, `.train()`, and `.eval()`. Therefore, it simplifies our life a lot!

Let's see how we can adapt our models with `nn.Module`:

### a. Signature of the class

Here, this class will inherit from `nn.Module`.  

In [ ]:
class MyModel(nn.Module):
    pass

### b. `__init__()`

As always, this is the **constructor** of the class, which in this case will define the architecture (layers) of the model. 

By using `nn.Module`, we do not need to include the parameters explicitly. We can define them through particular layers (which we will explore next),  and then organize them in a list to control their order.

All these components are grouped using an `nn.ModuleList` —  a special container designed to hold multiple `nn.Module` objects.

In [ ]:
class MyModel(nn.Module):

    def __init__(self):
        super().__init__() # Always do this when inheriting from nn.Module!

        # HERE COMES CODE
        l1 = None # Linear layer 2 inputs x 4 outputs 
        a1 = None # ReLU activation function
        l2 = None # Linear layer 4 inputs x 1 output 
        a2 = None # Sigmoid activation function
        l = [l1, a1, l2, a2]
        self.module_list = nn.ModuleList(l)

Notice that we put all layers in the `nn.ModuleList` object, which is just a list object composed of `nn.Module` items. This makes the code more readable and easier to follow. 

For now, do not worry about the different components shown here (`nn.Linear`, `nn.ReLU`, etc.). We will come back to them in a bit.

### c. `forward()` 

It defines the *computation*, i.e., it implements how input data passes through the network.  

In [ ]:
class MyModel(nn.Module):

    def __init__(self):
        super().__init__()
        l1 = nn.Linear(2, 4)
        a1 = nn.ReLU()
        l2 = nn.Linear(4, 1)
        a2 = nn.Sigmoid()
        l = [l1, a1, l2, a2]
        self.module_list = nn.ModuleList(l)

    def forward(self, X):
        X=None # HERE COMES CODE (loop through each layer, calculating its output at each step)
        return X

FYI, this is the same as the following, coding each operation explicitly:

In [ ]:
class MyModel2(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 4)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(4, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.sigmoid(x)
        return x

### Usage

Now we only need to instantiate the model and we can start using it!

In [ ]:
# Instantiation
model = MyModel()

# Input data (it needs to have 2 features)
X = torch.Tensor([[0.1, 0.7], [0.1, 0.7]])

# And we can just pass this data to the model
model(X)

<div class="alert alert-block alert-info">
    
🧐🧐 **PAY ATTENTION** 🧐🧐

As you can see, we were able to generate the output by passing the input to the model object in parentheses. That is, we did not need to call the `forward` method explicitly, because it is called implicitly.

💡**BUT**💡That is a capability of `nn.Module`, which we are inheriting from. See what happens if we don't inherit from this class and try to do the same:
</div>

In [ ]:
class MyModelNoModule(object): # See we don't have nn.Module inheritance

    def __init__(self):
        super().__init__()
        l1 = nn.Linear(2, 4)
        a1 = nn.ReLU()
        l2 = nn.Linear(4, 1)
        a2 = nn.Sigmoid()
        l = [l1, a1, l2, a2]
        self.module_list = nn.ModuleList(l)

    def forward(self, X):
        for f in self.module_list:
            X = f(X)
        return X

In [ ]:
model_2 = MyModelNoModule()
model_2(X)

It would only work if we explicitly called the `forward` method.

In [ ]:
model_2.forward(X)

> **Question**: Each time you run the cell where we instantiated and used the model you get a different output despite having the same input. Why do you think this is?
> 
> **Answer:**
> 

### Making the most out of `nn.Module`: Components of a Neural Network

Now that we understand what `nn.Module` is and why our model should inherit from it,  let’s look inside and explore the **key building blocks** that make up a neural network.

When defining the `__init__()` method, we saw that we can include elements such as:
- `nn.Linear` — layers that apply a linear transformation.
- `nn.ReLU`, `nn.Sigmoid`, etc. — activation functions that introduce non-linearity and generate output predictions.
- Other modules like `nn.Dropout`, `nn.BatchNorm1d`, etc. (next week!)

Each of these is itself a **subclass** of `nn.Module`, so they automatically register their own parameters (if they have any)  and integrate smoothly with the rest of PyTorch’s machinery.

Let's now go through these components one by one to understand what each does and how they work together within our model.

#### Layers

Layers in `PyTorch` include transformations from one layer of the neural network to the next. 

These transformations typically depend on certain weights $W$. In `PyTorch` once you define a layer with its **architecture**, i.e. the expected number of inputs and outputs, the weights are initialized to a random value.

**These weights are what we learn during training.**

In a standard neural network, we would typically use `nn.Linear`, which is the usual linear transformation of the incoming data. Specifically, given an input sample with `in_features`, this would yield $k$ `out_features` as: $h_k = \sum_{j=1}^{\mbox{in\_features}} x_j W_{jk} + b_k$ 

This class takes the following arguments:
- `in_features` and `out_features` are the sizes of each input and output sample, respectively.

In [ ]:
# 1 input -> 2 outputs
m = # HERE COMES CODE (a linear layer with 1 input and 2 outputs)
input = torch.tensor([1.0])
output = m(input)
print(output.size(), output)

> **Question**: This layer was able to produce an output simply by passing the input in parentheses. Why do you think this is?
>
>  **Answer:**
> 

#### Activations

- `nn.ReLU`

In [ ]:
m = None # HERE COMES CODE (ReLU activation function)
input = torch.tensor([-0.1, 0.4, -0.3, 0.1])
output = m(input)
output

- `nn.Sigmoid`

In [ ]:
m = # HERE COMES CODE (Sigmoid activation function)
input = torch.tensor([-0.9, 0., -5, 5])
output = m(input)
output

- `nn.Tanh`: Hyperbolic Tangent (Tanh) function

In [ ]:
m = None # HERE COMES CODE (Tanh activation function)
input = torch.tensor([-0.9, 0., -5, 5])
output = m(input)
output

- `nn.Softmax`: applies the softmax function to an n-dimensional input. This is the typical layer for multi-class problems!

In [ ]:
m = None # HERE COMES CODE (Softmax activation function)
input = torch.tensor([[-0.9, 0., -1, 1]])
output = m(input)
print(output)

print(output.sum())

> **Question**: Here again, we were able to produce an output simply by passing the input in parentheses. Why do you think this is?

## 2. Training the model

This is when the created model is fitted with respect to our data (normally the training set).  During this process, our cost function is minimized, and the parameters of our model are learned.

So far, we have been doing this for a given cost function (e.g. MSE) through a loop implementing stochastic gradient descent.

Now, since our model is a subclass of `nn.Module`, `PyTorch` already knows which parameters belong to it.   This allows us to fully leverage the `PyTorch` ecosystem — using its **built-in cost functions** and, more importantly, **optimizers** — to handle the entire training process automatically.

This means we no longer need to manually implement stochastic gradient descent.  Instead, we can use other and sometimes more flexible and effective optimizers (e.g., Adam), all of which directly access and update the parameters registered within our `nn.Module` model.

In [ ]:
# Let's use this model
class MyModel(nn.Module):

    def __init__(self):
        super().__init__()
        l1 = nn.Linear(2, 10)
        a1 = nn.ReLU()
        l2 = nn.Linear(10, 1)
        a2 = nn.Sigmoid()
        l = [l1, a1, l2, a2]
        self.module_list = nn.ModuleList(l)

    def forward(self, X):
        for f in self.module_list:
            X = f(X)
        return X

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import matplotlib.pylab as plt

X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)
X_train, X_report, y_train, y_report = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_report = scaler.transform(X_report)
X = scaler.transform(X)

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
X_report = torch.tensor(X_report, dtype=torch.float32)
y_report = torch.tensor(y_report, dtype=torch.float32).reshape(-1, 1)

plt.scatter(X[:,0], X[:,1], c=y, cmap="RdBu", edgecolors='w', alpha=1)
plt.xlabel("Feature 1", size=20)
plt.ylabel("Feature 2", size=20)
plt.show()

During training, as we have seen, we aim to find the model parameters (weights and intercepts) that minimize a **cost function** according to some optimization rule (e.g., stochastic gradient descent).

The good news is that, when our model is defined as a subclass of `nn.Module`, this task becomes **much simpler**.  

PyTorch automatically keeps track of all parameters in the model, allowing its built-in **cost functions** and **optimizers** to handle most of the heavy lifting for us.

Let's have a quick look at how this works in practice —  we will explore each component in detail right after.


In [ ]:
from torch import optim

torch.manual_seed(1234)
# 1- Instantiate the model

model = None # HERE COMES CODE 

# 2- Define the cost function through nn
cost_function = None # HERE COMES CODE  # This is the cost function
# 3- Define the optimization rule through optim module
optimizer = None # HERE COMES CODE (SGD with lr=0.01)  # pay attention to the first argument!

n_epochs = 100
train_costs = []
report_costs = []

# And this is the typical training loop!
for epoch in range(n_epochs):

    model.train() # Use this when training, typically at the start of each epoch!
    
    running_cost = 0
    accuracy_hist_train = 0
    randperm = torch.randperm(X_train.shape[0])
    for ii in randperm:
        x_batch, y_batch = X_train[ii], y_train[ii]

        # 4- compute outputs
        y_probs = None # HERE COMES CODE (just passing x_batch to the model)
        # 5- compute cost
        cost = None # HERE COMES CODE (cost function between y_batch and y_probs)
        # 6- Calculate gradients
        # HERE COMES CODE 
        # 7- Apply them to update the parameters
        # HERE COMES CODE 
        
        # 8- set the gradients to zero -- this is similar to the other day
        # HERE COMES CODE 
        
        running_cost += cost.item()

    train_costs.append(running_cost / X_train.shape[0])

    # Print loss every 10 epochs
    if epoch % 10 == 0:
        print(f"Epoch {epoch+1}: Train Loss = {train_costs[-1]:.4f}")

So as you can see, the new ingredients here are:

- Importing a cost function from `nn`. This is barely a new change.
- Include `.train()` at the beginning of each epoch. This tells PyTorch that the model is in **training mode**, allowing training-specific behaviors in certain parts of the network, such as dropout and batch normalization (we will cover these in detail in our next class!).
- Minimization of the cost function through an optimizer object. **This is a big and major advance! (Compare to previous lesson!)**

In [ ]:
# Create a grid of points covering the feature space
x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 1000),
    np.linspace(y_min, y_max, 1000)
)
grid = np.c_[xx.ravel(), yy.ravel()]
grid_tensor = torch.tensor(grid, dtype=torch.float32)

# Get model predictions on the grid
with torch.no_grad():
    probs = model(grid_tensor).numpy().reshape(xx.shape)

# Plot decision boundary
plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, probs, levels=50, cmap="RdBu", alpha=0.6)
cbar = plt.colorbar()
cbar.set_label("Predicted Probability", size=15)
cbar.ax.tick_params(labelsize=10)
# Overlay all data
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="RdBu", edgecolors='w', alpha=1)
plt.title("Decision Boundary", size=20)
plt.xlabel("Feature 1", size=15)
plt.ylabel("Feature 2", size=15)
plt.show()

## 3. Evaluating the model on the reporting set

<div class="alert alert-block alert-info">

𝐈𝐌𝐏𝐎𝐑𝐓𝐀𝐍𝐓‼️ 

Always start the evaluation part with `model.eval()`! This will be benefitial to disable certain uneeded parts during evaluation time. We will cover what these parts are in our next lab!

</div>

In [ ]:
model.eval()  # Important: always start with this!
report_cost = 0.0
correct = 0
total = 0

with torch.no_grad():  # No gradient tracking
    for ii in range(X_report.shape[0]):
        x_batch, y_batch = X_report[ii], y_report[ii]
        probs = model(x_batch)
        cost = cost_function(probs, y_batch)
        report_cost += cost.item() 
        predictions = (probs > 0.5).float() 
        correct += (predictions == y_batch).sum().item()
        total += y_batch.size(0)

avg_report_loss = report_cost / total
accuracy = correct / total

print(f"Reporting Loss: {avg_report_loss:.4f}, Reporting Accuracy: {accuracy:.2%}")

<div class="alert alert-block alert-info">

𝐈𝐌𝐏𝐎𝐑𝐓𝐀𝐍𝐓‼️ 

Technically, we did not need to evaluate each sample in the reporting set individually, especially for a small dataset. We could have passed the whole reporting set to the model at once.
</div>

In [ ]:
# Using the whole reporting set
from sklearn.metrics import accuracy_score

model.eval() # 
with torch.no_grad():
    probs = model.forward(X_report)
    report_cost = cost_function(probs, y_report)
    y_pred = (probs > 0.5).detach().numpy().flatten()
    
accuracy = accuracy_score(y_report.numpy(), y_pred)
print(f"Reporting Loss: {report_cost:.4f}, Reporting Accuracy: {accuracy:.2%}")

🚨🚨 But in practice, it is recommended to follow the same approach as in the training set when handling individual samples. This keeps your pipeline consistent and reduces the chance of confusion or bugs.

🚨🚨 In addition, as we will see later, when using batches (rather than single samples), this is both more memory- and computation-efficient, and it scales better for larger datasets and models.

### Cost functions

The module `nn` has a lot of implementations. Here are a few major and standard ones:

- `nn.MSELoss`: Mean squared error (squared L2 norm) between each element in the input x and target y. Cost function for **standard regression** tasks.
- `nn.BCELoss`: Binary Cross Entropy between the target and the input probabilities. Cost function for standard **binary classification** tasks.
- For **multi-class classification**, we should be using the categorical cross-entropy cost function. `nn` has its own implementation, but it handles raw outputs, instead of probabilities from the softmax activation function. As a result, for didactic reasons, in this course we will use the same implementation we already introduced in the previous lab

   $$-\sum_{c=1}^{l} y_{o,c}\,\log(p_{o,c}),$$

  where $l$ is the number of classes, $y_{o,c}$ is the one-hot encoded label for observation $o$, and $p_{o,c}$ is the predicted probability for class $c$ (after applying softmax). The log is the natural logarithm.


Here is its implementation inheriting from `nn.Module`:

In [ ]:
import torch
import torch.nn as nn

class CategoricalCrossEntropy(nn.Module):
    def forward(self, y_pred, y_true):
        cost = 0
        for ii in range(y_true.shape[1]):
            y_oc = y_true[:, ii]
            y_pc = y_pred[:, ii]
        
            cost -= y_oc*torch.log(y_pc.clamp_min(1e-12))

        return cost.mean(0) # This is just to average across all observations    

Let's suppose we have the following 5 class problem:

In [ ]:
from sklearn.preprocessing import OneHotEncoder

outputs = torch.randn(3, 5)  # 3 observations, 5 output values each

# Convert outputs into probabilities
probs = nn.Softmax(dim=1)(outputs)

# Suppose the following labels for these observations
labels = np.array([1, 4, 2]).reshape(-1, 1)

encoder = OneHotEncoder(
    categories=[[0, 1, 2, 3, 4]],
    sparse_output=False
)

y_obs = torch.tensor(
    encoder.fit_transform(labels),
    dtype=torch.float32
)

In [ ]:
cost = None # HERE COMES CODE (just the Cross entropy loss)
print(cost(probs, y_obs))

<div class="alert alert-block alert-info">

🧐 **Pay attention!!**

Do you know why we call `criterion(y_pred, y_true)` directly?

Since cost functions like `nn.MSELoss`, `nn.BCELoss` and this one above, `CategoricalCrossEntropy` are also subclasses of `nn.Module`, calling them as functions automatically runs their `forward()` methods, just as with layers or models:

</div>

In [ ]:
cost.forward(probs, y_obs)

If you are interested in using the categorical cross-entropy cost function implementation provided by `pytorch`, plase visit: https://docs.pytorch.org/docs/2.14/generated/torch.nn.CrossEntropyLoss.html#crossentropyloss. But again, keep in mind that its outputs will not be softmax values, but the raw ones.

You can check the [cost-functions documentation](https://docs.pytorch.org/docs/stable/nn.html#loss-functions) for more cost functions.

### Optimizers

We will use `torch.optim`, which is a package implementing various optimization algorithms.

To use `torch.optim`, you need to define an optimizer object, which will hold the current state and will update the parameters based on the computed gradients.

This is the new part:

```python
optimizer = optim.Adam(model.parameters())
...
optimizer.step()
...
optimizer.zero_grad()
```
which replaced the part:

```python
with torch.no_grad():
    self.W_1 -= self.learning_rate * self.W_1.grad
    self.b_1 -= self.learning_rate * self.b_1.grad
    self.W_2 -= self.learning_rate * self.W_2.grad
    self.b_2 -= self.learning_rate * self.b_2.grad

    # Zero gradients
    self.W_1.grad.zero_()
    self.b_1.grad.zero_()
    self.W_2.grad.zero_()
    self.b_2.grad.zero_()
```

<div class="alert alert-block alert-info">

🫠 **DO NOT PANIC!!**

You see that we have placed `optimizer.zero_grad()` after calculating and applying the gradients. This was intended to mirror what we have been doing in previous lessons.

However, PyTorch recommends clearing the gradients **before** calculating new ones, i.e.:

```python
optimizer.zero_grad()
cost.backward()
optimizer.step()
```

Both approaches are valid, as long as gradients are cleared before the next backward pass. It is mostly a matter of convention.

</div>

The following are the same optimizers that you can find for `MLPClassifier` and `MLPRegressor` in `scikit-learn`

- `torch.optim.SGD`:  Stochastic Gradient Descent. Widely used in practice for both simple and deep models. It may lead to better generalization than adaptive methods (like Adam; see next) due to its flexibility to tune parameters.

- `torch.optim.Adam`: Optimizer method that adapts the learning rate. Therefore, it typically requires minimal tuning. A default for deep learning.

- `torch.optim.LBFGS`:  A quasi-Newton method that approximates the inverse Hessian matrix. It is useful for smaller datasets and optimization problems requiring very precise convergence. Less commonly used in deep learning. 

`PyTorch` includes many more optimizers beyond the ones we've covered. You can find a more extensive list in [PyTorch's optimization algorithms](https://pytorch.org/docs/stable/optim.html#algorithms).

In practice, **SGD and Adam are by far the most commonly used**, and we will be using these two for the rest of the module on neural networks.

<div class="alert alert-block alert-warning">

**NOTE!!!**
    
`torch.optim.LBFGS` requires a special mechanism called a *closure* during optimization, which makes it more cumbersome to use. If you're curious, you can read more about it in the [official PyTorch documentation](https://pytorch.org/docs/stable/optim.html#optimizer-step-closure).

</div>


<div class="alert alert-block alert-info">

𝐈𝐌𝐏𝐎𝐑𝐓𝐀𝐍𝐓‼️ 

The **first argument** to an optimizer is the set of **model parameters** — these are the values we want to update during training.

Because our model **inherits from `nn.Module`**, we can simply call `model.parameters()` to access them automatically.  
Otherwise, we would have to manually collect and pass all parameters, which is error-prone and cumbersome.

**This is yet another reason why we always define our models as subclasses of `nn.Module`!**

</div>

## 4. Improving Training with Batches using `DataLoader`

Up to now, we have been training our models by updating the weights after each individual training sample. This is the main behavior behind **Stochastic Gradient Descent**.

However, in practice, this can make training problematic due to:

- Slower and less stable convergence.
- Noisy gradient updates
  
We can improve this by using **batches** of training samples, i.e. by considering several samples simultaneously to update our parameters.

We will see that this is very easy to do in `PyTorch` through `DataLoader`.


In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# Define batch size
batch_size = 16

# Wrap the training dataset in a TensorDataset and DataLoader
train_dataset = None # HERE COMES CODE (Use TensorDataset with X_train and y_train)
train_loader = None # HERE COMES CODE (Use DataLoader with train_dataset, batch_size=batch_size, shuffle=True)

# Wrap the reporting dataset in a TensorDataset and DataLoader
report_dataset = None # HERE COMES CODE (Use TensorDataset with X_report and y_report) 
report_loader = None # HERE COMES CODE (Use DataLoader with report_dataset, batch_size=batch_size, shuffle=False!)

In [ ]:
from torch import optim

model = MyModel()

cost_function = nn.BCELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

n_epochs = 100
train_costs = []
report_costs = []

for epoch in range(n_epochs):
    model.train() # Use this when training, typically at the start of each epoch!

    running_cost = 0
    accuracy_hist_train = 0
    
    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()
        y_probs = model(x_batch)
        cost = cost_function(y_probs, y_batch)
        cost.backward()
        optimizer.step()
        running_cost += cost.item()

    train_costs.append(running_cost / len(train_loader))

    # Print loss every 10 epochs
    if epoch % 10 == 0:
        print(f"Epoch {epoch+1}: Train Loss = {train_costs[-1]:.4f}")

With just a few lines of code, we have switched from stochastic to batch training!

`DataLoader` gives us clean and efficient batch processing, and prepares us to handle larger datasets in future labs.



In [ ]:
model.eval()  # Important: disable dropout, batchnorm, etc.
report_cost = 0.0
correct = 0
total = 0

with torch.no_grad():  # No gradient tracking
    for x_batch, y_batch in report_loader:
        probs = model(x_batch)
        cost = cost_function(probs, y_batch)
        report_cost += cost.item() * x_batch.size(0)  
        predictions = (probs > 0.5).float() 
        correct += (predictions == y_batch).sum().item()
        total += y_batch.size(0)

avg_report_loss = report_cost / total
accuracy = correct / total

print(f"Reporting Loss: {avg_report_loss:.4f}, Reporting Accuracy: {accuracy:.2%}")

In [ ]:
from sklearn.metrics import accuracy_score
# Get model predictions on the reporting set
with torch.no_grad():
    probs = model.forward(X_report).detach().numpy().flatten()
    y_pred = (probs > 0.5)
print(accuracy_score(y_report.numpy(), y_pred))

## 5. Summary

- **`nn.Module`** provides an efficient and consistent framework to build neural networks.  
  It automatically registers parameters, connects layers, integrates with loss functions and optimizers,  
  and makes training pipelines clean and scalable.

- Use `.train()` at the beginning of each epoch during training to allow training-specific components. Conversely, use `.eval()` during evaluation to disable these components.

- **Cost functions** from `torch.nn` simplify the computation of losses and work seamlessly with models built using `nn.Module`.

- **Optimizers** from `torch.optim` automatically update model parameters (retrieved through `model.parameters()`),  
  allowing flexible and reliable training beyond manual gradient descent.

- **`DataLoader`** allows for efficient **batch processing**, shuffling, and iteration over large datasets —  
  essential for scalable and reproducible training.


## 📝 6. Exercises

### 6.1 Exercise — Adapting to Multiclass and Regression

Replicate the same architecture and training loop from class, but apply it to the following tasks:

1. **Multiclass Classification**
   - Use `sklearn.datasets.make_classification` 
   - Modify the final layer to output `n_classes` units
   - Use `CategoricalCrossEntropy` provided above
   - Use a softmax activation function

2. **Regression**
   - Use `sklearn.datasets.make_regression`
   - Final layer should output a single unit
   - Use `nn.MSELoss`
   - No activation function at the output layer

In both cases, use a batch size other than 1. 

🧠 *Hint*: Each task needs a different loss function and output layer. Be careful with data shapes and target formats.

In [ ]:
#YOUR CODE

### 6.2 Exercise  — Compare Optimizers and Batch Sizes

Train the binary classification model from class using different optimizers:

- `torch.optim.SGD`
- `torch.optim.Adam`

Keep the same architecture, data, and number of epochs for each run. Compare:

- Final training loss
- Training stability (does one converge more smoothly than the other?)
- Tuning accuracy (optional; split the training data into fitting and tuning sets for this comparison, keeping the reporting set for final evaluation).

Also, experiment with different **batch sizes** (e.g., 1, 8, 32, 64) and compare:

- Training performance (e.g., convergence speed, final loss)
- Computational time

**Reflect**: What do you notice when using very small vs. very large batches? Which optimizer converges fastest? Which one performs best?

In [ ]:
#YOUR CODE